[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/hanging_chain/hanging_chain.ipynb)

# Newton's Method for a Hanging Chain

This example computes the equilibrium shape of a hanging chain by Newton's method. A chain of beads joined by stiff springs hangs between two pins at the same height, settling at rest into the shape of minimum potential energy where spring tension balances gravitational pull. With stiff springs, that equilibrium curve lies close to the inextensible catenary.

All interior bead positions form a single field on the chain's complex, with link vectors taken by the coboundary `topology.d[0]` and gravitational work paired via `weight & beads`. Applying `derivative` once and twice to `energy` yields the gradient form on chain-wide displacements and the symmetric curvature form on pairs of displacements, so `curvature.solve(gradient)` updates every bead in one Newton step while `gradient.adjugate()(antiscalar)` reads the unbalanced force at every bead across the work pairing.

In matrix notation, stacking the $N$ bead coordinates into $\mathbf{q} = (x_1, y_1, \dots, x_N, y_N)^\top \in \mathbb{R}^{2N}$ requires differentiating each link vector $\Delta\mathbf{r}_e$ into the $2 \times 2$ tangent stiffness block $\mathbf{K}_e = \frac{k}{\ell_0}\!\left[\hat{\mathbf{t}}_e\hat{\mathbf{t}}_e^\top + \left(1 - \frac{\ell_0}{\|\Delta\mathbf{r}_e\|}\right)(\mathbf{I}_2 - \hat{\mathbf{t}}_e\hat{\mathbf{t}}_e^\top)\right]$ with $\hat{\mathbf{t}}_e = \Delta\mathbf{r}_e / \|\Delta\mathbf{r}_e\|$, scattering those blocks into the $2N \times 2N$ block-tridiagonal Hessian $\mathbf{H}(\mathbf{q})$ alongside the residual vector $\mathbf{g}(\mathbf{q}) = \nabla_{\mathbf{q}} E(\mathbf{q})$, and solving $\mathbf{H}(\mathbf{q})\,\Delta\mathbf{q} = -\mathbf{g}(\mathbf{q})$ at each step.

In [ ]:
# Install numga and its examples when running in Colab.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[jax,examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline

from collections.abc import Iterator

import jax
import numpy as np
from scipy.optimize import brentq

from numga import stack
from numga.algebras import VGA2D
from numga.backend.jax import JaxContext, derivative
from numga.topology import Topology
from examples.util.animation import save_figure
from examples.mechanics.hanging_chain import render


ga = VGA2D
mv = JaxContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
# Planar displacements are intrinsic and forces are extrinsic, paired via work `force & displacement`.
Point = ga.gatype.antivector()                      # Antivector-
Force = ga.gatype.vector()                          # Vector+
unit_x, unit_y = mv.x.dual(), mv.y.dual()           # [] Point-, Point-
antiscalar = mv.scalar([1]).dual()                  # [] Antiscalar-

## Potential energy of the chain

Each spring stores elastic energy proportional to the square of its stretch beyond rest length, while gravitational potential energy is minus the work `weight & beads` done by gravity on the beads' displacements from the midpoint between the pins. On the chain's 1D complex, the coboundary `topology.d[0]` takes every link vector from the interior bead positions together with the two fixed boundary pins. Because the energy depends only on link lengths and gravitational work from a fixed origin, the flat plane's intrinsic displacements and extrinsic forces suffice without a projective weight coordinate.

In [ ]:
n_beads = 31
span = 2.0
length = 2.5
stiffness = 1e3
bead_mass = 0.1
gravity = 9.81

In [ ]:
topology = Topology.chains([n_beads + 2])
V, E = topology.n_elements
left_pin, right_pin = unit_x * (-span / 2), unit_x * (span / 2)                 # [] Point-, Point-
pins = stack([left_pin, right_pin]).field()                                     # Point-[2]
weight = mv.y * (-bead_mass * gravity)                                          # [] Force+
rest_length = length / E
# Split the coboundary d[0] across the free interior beads and the two fixed exterior pins.
interior = topology.reading(topology.interior[0], 0)                            # Scalar+[V_free <- V]
exterior = topology.reading(topology.exterior[0], 0)                            # Scalar+[2 <- V]
bead_ends = topology.d[0] * interior.adjugate()                                 # Scalar+[E <- V_free]
pin_ends = topology.d[0] * exterior.adjugate()                                  # Scalar+[E <- 2]
anchors = pin_ends * pins                                                       # Point-[E]


def energy(beads: Point) -> Scalar:
    """Total potential energy sums the spring stretch energy less the gravitational work across the chain (`[...] Scalar+`)."""
    links = bead_ends * beads + anchors                                         # [...] Point-[E]
    stretch = links.norm() - rest_length                                        # [...] Scalar+[E]
    spring_energy = stiffness / (2 * rest_length) * stretch.squared()           # [...] Scalar+[E]
    return spring_energy.sites.sum() - (weight & beads).sites.sum()

## Stretched initial arc

Newton's method steps inverse to the energy's curvature, and a spring has positive-definite transverse stiffness only under tension because a compressed link buckles sideways. Starting the beads evenly spaced along a circular arc between the pins stretches every link slightly beyond its rest length so the initial curvature is positive definite everywhere.

In [ ]:
# Half-angle subtended by the pins from the lowest point of the starting arc.
arc_half_angle = 1.3

In [ ]:
radius = span / (2 * np.sin(arc_half_angle))
centre = unit_y * (radius * np.cos(arc_half_angle))                             # [] Point-
angles = np.linspace(-arc_half_angle, arc_half_angle, V)[1:-1]                  # [n_beads]
rotors = (mv.xy * (-angles / 2)).exp()                                          # [n_beads] Rotor+
initial_beads = (centre + (rotors >> (-radius * unit_y))).field()               # [] Point-[V_free]

## Newton's method on the whole chain

Differentiating `energy` once with respect to the bead field yields the gradient form on chain-wide displacements, and differentiating a second time yields the symmetric bilinear curvature form coupling each bead to itself and its immediate neighbours. Solving `curvature.solve(gradient)` finds the simultaneous displacement of all beads that cancels the gradient to first order, and compiling the step with `jax.jit` fuses the energy evaluation, both derivatives, and the dense linear solve.

In [ ]:
@jax.jit
def newton_step(beads: Point) -> Point:
    """One Newton step updates the bead field by solving the linearised equilibrium system (`[] Point-[V_free]`)."""
    gradient = derivative(energy)(beads)                                        # [] Scalar+ <- Point-[V_free]
    curvature = derivative(derivative(energy))(beads)                           # [] Scalar+ <- (Point-[V_free], Point-[V_free])
    return beads - curvature.solve(gradient)                                    # [] Point-[V_free]


initial_gradient = jax.jit(derivative(energy))(initial_beads)                   # [] Scalar+ <- Point-[V_free]
initial_curvature = jax.jit(derivative(derivative(energy)))(initial_beads)      # [] Scalar+ <- (Point-[V_free], Point-[V_free])

print(initial_gradient)
print(initial_curvature)

## Quadratic convergence of the iterates

Evaluating the energy derivative across the batch of iterates produces the gradient form at every step in one call, and `gradient.adjugate()(antiscalar)` pulls the unit antiscalar back across the work pairing to give the unbalanced force on every bead. Once that residual force drops below a single bead's weight, each Newton step squares the remaining error until the chain reaches equilibrium at round-off.

In [ ]:
n_steps = 8

In [ ]:
def newton_iterates(beads: Point, n_steps: int) -> Iterator[Point]:
    """Yields the initial bead field followed by `n_steps` Newton iterates (`[] Point-[V_free]`)."""
    yield beads
    for _ in range(n_steps):
        beads = newton_step(beads)
        yield beads


iterates = stack(newton_iterates(initial_beads, n_steps))                       # [n_steps + 1] Point-[V_free]
# Pull the unit antiscalar back across the work pairing `force & displacement` to obtain the unbalanced force field.
unbalanced_forces = -jax.jit(derivative(energy))(iterates).adjugate()(antiscalar)   # [n_steps + 1] Force+[V_free]
residual = unbalanced_forces.norm() / (bead_mass * gravity)                     # [n_steps + 1] Scalar+[V_free]

## Comparison with the inextensible catenary

An inextensible chain of uniform density hangs along a hyperbolic cosine curve whose arc length between the two pins fixes its bottom radius of curvature, with a vertical offset placing both pins at height zero. Because finite stiffness allows the springs to stretch by one to two percent under the chain's weight, the equilibrium bead positions hang slightly below the inextensible catenary of the same rest length, approaching it inversely with spring stiffness.

In [ ]:
n_curve_samples = 200

In [ ]:
def catenary_radius(span: float, length: float) -> float:
    """Root-finding on the catenary arc-length relation determines the radius of curvature at the lowest point."""
    return brentq(lambda radius: 2 * radius * np.sinh(span / (2 * radius)) - length, span / 20, 100 * span)


def catenary_height(horizontal: Scalar, bottom_radius: float, span: float) -> Scalar:
    """A hyperbolic cosine profile gives the catenary height at each horizontal coordinate with both pins at zero (`[...] Scalar+`)."""
    return bottom_radius * ((horizontal / bottom_radius).cosh() - np.cosh(span / (2 * bottom_radius)))


bottom_radius = catenary_radius(span, length)
horizontal = mv.scalar(np.linspace(-span / 2, span / 2, n_curve_samples)[:, None])  # [n_curve_samples] Scalar+
catenary = unit_x * horizontal + unit_y * catenary_height(horizontal, bottom_radius, span)  # [n_curve_samples] Point-
save_figure(render.iterates(iterates, residual, catenary, pins), "hanging_chain");

Light grey polylines trace the early Newton iterates from the circular arc down to the blue equilibrium chain with marked beads, which hangs just beneath the dashed red inextensible catenary on the left while the maximum unbalanced force on the right drops below round-off in six steps.

In [ ]:
# checks
max_residual = residual.batch().max(axis=-1)                                    # [n_steps + 1] Scalar+
assert bool(max_residual[-1] < 1e-9)
converging = max_residual[:-1] < 1
assert bool((max_residual[1:][converging] < max_residual[:-1][converging].squared() + 1e-9).all())
assert bool(energy(iterates[-1]) < energy(iterates[0]))
equilibrium_beads = iterates[-1].batch()                                        # [n_beads] Point-
catenary_heights = catenary_height(mv.x & equilibrium_beads, bottom_radius, span)   # [n_beads] Scalar+
stretch_tolerance = length * n_beads * bead_mass * gravity / stiffness
assert bool((((mv.y & equilibrium_beads) - catenary_heights).abs() < stretch_tolerance).all())